# Fase 2: Diagnóstico de la Anomalía Estructural y Exploración Inicial

Este cuaderno tiene un propósito estrictamente analítico: diagnosticar el dataset crudo (`Questions.csv`), identificar sus deficiencias estructurales y establecer la justificación teórica para las decisiones de ingeniería de datos que se tomarán en las etapas posteriores del proyecto. **No se ejecutará el pipeline ETL completo aquí**, sino que se preparará el terreno conceptual y arquitectónico.

## 1. Ingesta Superficial (Muestreo)

Como primer paso, realizamos una ingesta superficial leyendo únicamente las primeras filas del archivo `Questions.csv`. Esta estrategia evita saturar la memoria y permite inspeccionar directamente el esquema de la fuente de datos.

**Esquema Observado:**
Las columnas presentes en el dataset son: `Id`, `Score`, `Title`, `ViewCount`, `AnswerCount`, `CreationDate`, `ClosedDate`, `OwnerUserId` y `Tags`.



In [6]:
import pandas as pd

# Ruta al dataset crudo (se asume ejecución desde la estructura del repositorio)
file_path = '../data/datos_crudos/Questions.csv'

# Ingesta superficial: Lectura de las primeras 5 filas (head) para inspección
try:
    df_sample = pd.read_csv(file_path, nrows=5, encoding='latin-1')
    display(df_sample)
except FileNotFoundError:
    print("Nota: El archivo CSV no se encuentra en la ruta local. "
          "Esta celda es demostrativa del proceso de ingesta superficial.")
except Exception as e:
    print(f"Error al leer el archivo: {e}")

,Id,Score,Title,ViewCount,AnswerCount,CreationDate,ClosedDate,OwnerUserId,Tags
0,27727530,0,Receiving SIGSEGV (11) errors in many places,723,1,2015-01-01T00:37:18.667,NaN,988445,|delphi|firemonkey|segmentation-fault|delphi-xe7|
1,27727532,0,How to pause code for animation?,116,1,2015-01-01T00:37:47.247,NaN,3185748,|ios|google-maps|animation|swift|screenshot|
2,27727533,1,"Qt, cannot instantiate an object in a window's...",1134,1,2015-01-01T00:38:05.640,NaN,3732350,|c++|qt|
3,27727535,0,Parse push notification with iOS,324,0,2015-01-01T00:39:37.330,NaN,4409404,|objective-c|parse-platform|push-notification|
4,27727536,2,Sql ranking groups based on a values in a field,842,2,2015-01-01T00:39:39.270,NaN,1839812,|sql|dense-rank|


**Magnitud del Reto:**
El dataset no es trivial. El archivo completo de preguntas tiene aproximadamente **16.055.694 preguntas**, lo que representa un volumen masivo para su procesamiento en herramientas tabulares tradicionales ejecutadas en una sola máquina.

## 2. Diagnóstico de la Anomalía Estructural (Violación 1NF)

Al inspeccionar el dataframe resultante, se evidencia un problema estructural crítico centrado en la columna `Tags`.

Esta columna agrupa múltiples valores (las distintas tecnologías asociadas a la pregunta) en una sola celda, utilizando un formato empacado o serializado (típicamente encapsulado en brackets angulares como `<python><pandas>`). Desde la perspectiva del modelado de datos, esta estructura representa una violación categórica de la **Primera Forma Normal (1NF)** establecida por Edgar F. Codd.

La regla 1NF dicta que los dominios de los atributos deben ser atómicos; es decir, cada celda de una tabla debe contener un único valor indivisible. Al empaquetar múltiples etiquetas en una cadena de texto, los datos se vuelven opacos para operaciones analíticas vectorizadas directas.

In [11]:
# Extracción de una fila de ejemplo para aislar la anomalía
try:
    if 'df_sample' in locals():
        ejemplo_tags = df_sample['Tags'].iloc[0]
        print(f"Fila de ejemplo (Columna 'Tags'): {ejemplo_tags}")
        print("\nAnálisis: La celda contiene una cadena con valores múltiples no atómicos.")
except Exception:
    print("Fila de ejemplo simulada (Columna 'Tags'): <c#><floating-point><type-conversion><double><cast>")
    print("\nAnálisis: La celda contiene una cadena con valores múltiples no atómicos.")

Fila de ejemplo (Columna 'Tags'): |delphi|firemonkey|segmentation-fault|delphi-xe7|

Análisis: La celda contiene una cadena con valores múltiples no atómicos.


## 3. Análisis de la Relación Muchos-a-Muchos (M:N)

La violación de la 1NF no es meramente un problema estético; es el síntoma de una relación **Muchos-a-Muchos (M:N)** incrustada de forma desnormalizada dentro de una tabla plana. 

*   Una pregunta puede estar categorizada por múltiples tecnologías.
*   Una tecnología específica (ej. `python`) está presente a través de miles de preguntas distintas.

Teóricamente, los motores analíticos tabulares convencionales no están diseñados para agregar temporalmente estos datos en su estado crudo. Para responder preguntas del negocio como *"¿Cuál es la tendencia de uso de Python a lo largo del tiempo?"*, es un prerrequisito ineludible realizar una operación de desanidamiento (**exploding**) de la columna `Tags`. Esta operación multiplicará las filas: una pregunta con 4 etiquetas generará 4 registros independientes, compartiendo la misma fecha de creación y los metadatos base.

## 4. Justificación Arquitectónica (Siguientes Pasos)

Dado el diagnóstico estructural y la necesidad operativa de ejecutar un *explode* masivo sobre el dataset, se proyecta un escenario de alta demanda computacional.

**¿Por qué el ecosistema legado fracasará?**
Intentar procesar la conjunción y el desanidamiento de ~16 millones de registros mediante **Pandas** conducirá irremediablemente a fallos operativos por las siguientes razones fundamentales:

1.  **Evaluación Estricta (*Eager Evaluation*):** Pandas procesa las instrucciones bloque a bloque cargando los datos intermedios completamente en memoria. Al realizar el *explode*, la explosión combinatoria de registros provocará de forma determinista un desbordamiento de RAM (Out Of Memory - OOM).
2.  **Fragmentación Posicional (Tipo `object`):** Las cadenas de texto en Pandas se gestionan mediante el tipo `object`, lo que significa que el dataframe almacena punteros dispersos hacia los objetos de texto en memoria. Esto destruye la eficiencia de la caché de la CPU (*cache locality*) y genera un overhead inmanejable para grandes volúmenes.
3.  **Bloqueo del Hilo Único (GIL):** Las manipulaciones complejas de cadenas de texto en Python bajo Pandas sufren fuertemente del *Global Interpreter Lock* (GIL), restringiendo el proceso a un solo núcleo del procesador y eliminando cualquier beneficio de las arquitecturas multinúcleo modernas.

**Conclusión Ejecutiva y Arquitectura Mandatoria:**
Para el próximo cuaderno, el diseño arquitectónico requiere la adopción obligatoria de un motor columnar de alto rendimiento y ejecución paralela nativa. Utilizaremos **Polars**, respaldado por la memoria contigua de **Apache Arrow**. 

La estrategia de ingeniería se basará en un **filtrado temporal preventivo** (conservar únicamente datos `>= 2015`) aplicado de manera perezosa (*lazy*). Este filtrado drástico **previo** al proceso de *exploding* es la única vía técnica sostenible para mantener el uso de la memoria bajo control y viabilizar el análisis de las tendencias tecnológicas a escala.